<a id="inizio"></a>
# 08 · pandas: operazioni sui DataFrame e tra DataFrame · Soluzioni

<div style="background:rgba(96,125,139,0.08); border-left:4px solid #607d8b; padding:12px 16px; border-radius:4px; margin:8px 0 12px 0;">

**Blocco 3 · Analisi dei dati con pandas e visualizzazione · Giornata 2 · Aula Avanzata &nbsp;·&nbsp; ⏱ ~115 min**

Il grosso del lavoro di tutti i giorni: prendere le righe giuste, togliere quello che non serve, aggiungere quello che manca, raggruppare e incrociare con un'altra tabella.

**In questo notebook impariamo a**

- selezionare righe e colonne con `loc`, `iloc` e i filtri
- pulire valori mancanti e duplicati e creare colonne nuove
- aggregare con `groupby` e unire tabelle con `merge` e `concat`

Prima di questo: [07 · pandas: DataFrame e import dei dati](07_Pandas_import_dati.ipynb) &nbsp;·&nbsp; Dati: `../Dati/letture_pod_2025.csv`, `../Dati/bolletta_esempio.xlsx`, `../Dati/utility.db`, `../Dati/U.S. Electricity Prices.csv`

</div>

<a id="indice"></a>
**Indice**

- [1. Selezionare](#sez-1)
    - [1.1 Righe per etichetta: loc](#sez-1-1)
    - [1.2 Righe per posizione: iloc](#sez-1-2)
    - [1.3 Filtrare con una condizione](#sez-1-3)
- [2. Pulire](#sez-2)
    - [2.1 Correggere un valore](#sez-2-1)
- [3. Trasformare](#sez-3)
- [4. Aggregare](#sez-4)
- [5. Unire](#sez-5)
    - [5.1 Impilare tabelle: concat](#sez-5-1)
- [6. Data Wrangler](#sez-6)
- [7. Esercizi](#sez-7)
    - [Esercizio 8.1 · Il POD con la lettura sballata](#es-8-1)
    - [Esercizio 8.1 bis · I prezzi del Texas](#es-8-1-bis)
    - [Esercizio 8.2 · Bolletta per cliente](#es-8-2)
    - [Esercizio 8.2 bis · Clienti senza letture](#es-8-2-bis)

In [ ]:
# librerie usate in questo notebook
import sqlite3

import pandas as pd

<a id="sez-1"></a>
## 1. Selezionare

Ripartiamo dal file del fornitore del notebook precedente: le letture mensili di sei POD, per
fascia. Lo leggiamo con i parametri che ci erano costati un errore a testa: separatore `;`,
virgola decimale, codifica latin-1.

In [ ]:
letture = pd.read_csv("../Dati/letture_pod_2025.csv", sep=";", decimal=",", encoding="latin-1")
letture.head()

Prima domanda di ogni analisi: quali colonne ci servono. Una colonna tra parentesi quadre è
una Series; una lista di colonne (doppie quadre) è un DataFrame più stretto.

In [ ]:
letture["kwh"].head()

In [ ]:
letture[["pod", "kwh"]].head()

<a id="sez-1-1"></a>
### 1.1 Righe per etichetta: loc

Per prendere una riga per nome serve un indice parlante. L'anagrafica dei clienti, nel file
Excel della bolletta, ha un POD per riga: con `set_index` lo mettiamo come indice.

In [ ]:
anagrafica = pd.read_excel("../Dati/bolletta_esempio.xlsx", sheet_name="Anagrafica")
anagrafica_pod = anagrafica.set_index("pod")
anagrafica_pod

`pod` non è più una colonna: è l'indice, in grassetto a sinistra. Adesso `loc` prende una
riga per etichetta, come CERCA.VERT prende una riga per chiave.

In [ ]:
anagrafica_pod.loc["IT001E45678901"]

Con due argomenti, riga e colonna, `loc` restituisce il singolo valore. Con due liste, un
pezzo di tabella.

In [ ]:
anagrafica_pod.loc["IT001E45678901", "cliente"]

In [ ]:
anagrafica_pod.loc[["IT001E12345678", "IT001E45678901"], ["cliente", "comune"]]

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Da `anagrafica_pod`, leggi con `loc` il comune del POD `IT001E23456789` e salvalo in
`comune_lavanderia`.

</div>

In [ ]:
comune_lavanderia = anagrafica_pod.loc["IT001E23456789", "comune"]
comune_lavanderia

In [ ]:
# Verifica: esegui senza modificare
assert comune_lavanderia == "Lodi", "❌ loc[etichetta della riga, nome della colonna]"
print("✅ Tutto corretto")

<a id="sez-1-2"></a>
### 1.2 Righe per posizione: iloc

`iloc` conta le righe da 0, come una lista, e ignora le etichette. Serve per sbirciare:
la prima riga, le prime tre, l'ultima.

In [ ]:
letture.iloc[0]

In [ ]:
letture.iloc[0:3]

Lo slicing è quello delle liste: fine esclusa, `-1` è l'ultima. La regola per scegliere:
nome → `loc`, posizione → `iloc`. Quasi sempre vorremo `loc`, o un filtro.

<a id="sez-1-3"></a>
### 1.3 Filtrare con una condizione

Un filtro è la condizione dell'`if`, applicata a tutta la colonna in un colpo. Il risultato è
una Series di `True` e `False`, una per riga: la chiamiamo maschera.

In [ ]:
mask = letture["kwh"] > 1300
mask.head()

`sum()` su una maschera conta i `True`. E la maschera tra parentesi quadre tiene solo le
righe `True`.

In [ ]:
mask.sum()

In [ ]:
letture[mask]

Quattro letture sopra i 1300 kWh: tre dello Studio Dentistico, nei mesi freddi, e una del
Panificio a luglio da 5785 kWh. Un panificio che a luglio consuma quattro volte un dentista
in gennaio: teniamola d'occhio.

Due condizioni si combinano con `&` (e) e `|` (oppure); `~` nega. Ogni condizione va tra
parentesi.

In [ ]:
mask_f1_alte = (letture["fascia"] == "F1") & (letture["kwh"] > 1300)
letture[mask_f1_alte]

<div style="background:rgba(217,83,79,0.12); border-left:4px solid #d9534f; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**⚠️ Attenzione**

Senza parentesi, `letture["fascia"] == "F1" & letture["kwh"] > 1300` dà un errore o un
risultato sbagliato in silenzio: `&` ha la precedenza sui confronti. E in pandas si usano
`&`, `|`, `~`, mai `and`, `or`, `not`: quelli funzionano su un `True` solo, non su una colonna.

</div>

In [ ]:
mask_f1 = letture["fascia"] == "F1"
letture[~mask_f1].head(4)

Quando i valori ammessi sono più di due, `.isin` con una lista evita una catena di `|`.

In [ ]:
mask_due_pod = letture["pod"].isin(["IT001E12345678", "IT001E45678901"])
letture[mask_due_pod].head()

La maschera va anche dentro `loc`, con il nome di una colonna: `loc[maschera, colonna]`
legge solo quella colonna delle righe filtrate. Qui la media delle letture in F1.

In [ ]:
round(letture.loc[mask_f1, "kwh"].mean(), 1)

Chi viene da SQL può scrivere la condizione come testo con `query`. Stesso risultato della
maschera: lo mostriamo perché capita di trovarlo nel codice dei colleghi.

In [ ]:
letture.query("fascia == 'F1' and kwh > 1300")

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Costruisci la maschera `mask_notte` per le letture in fascia F3 sopra i 700 kWh (due
condizioni) e metti le righe filtrate in `notturne`. Quante sono?

</div>

In [ ]:
mask_notte = (letture["fascia"] == "F3") & (letture["kwh"] > 700)
notturne = letture[mask_notte]
notturne

In [ ]:
# Verifica: esegui senza modificare
assert len(notturne) == 3, "❌ Due condizioni tra parentesi, unite da &"
assert set(notturne["fascia"]) == {"F3"}, "❌ Devono restare solo righe in fascia F3"
print("✅ Tutto corretto")

<div style="background:rgba(224,168,0,0.14); border-left:4px solid #e0a800; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**📌 Ricorda**

- Nome → `loc`, posizione → `iloc`.
- Filtro in due passi: prima la maschera, poi `df[mask]`.
- Più condizioni: parentesi, `&`, `|`, `~`.

</div>

<a id="sez-2"></a>
## 2. Pulire

I dati veri arrivano con buchi e doppioni. Partiamo da un contatore che per due giorni non ha
trasmesso e che un giorno ha trasmesso due volte.

In [ ]:
contatore = pd.DataFrame({
    "giorno": ["2025-03-01", "2025-03-02", "2025-03-02", "2025-03-03", "2025-03-04", "2025-03-05", "2025-03-06"],
    "kwh": [12.3, 11.8, 11.8, None, 12.9, None, 13.4],
})
contatore

`isna()` segna i valori mancanti (pandas li mostra come `NaN`); `.sum()` li conta per colonna.
È la prima cosa da guardare dopo `info()`.

In [ ]:
contatore.isna().sum()

Due strade: buttare le righe incomplete con `dropna`, o riempire i buchi con `fillna`.
Dipende dalla domanda: per un totale mensile, un giorno a zero è un errore e un giorno alla
mediana è una stima onesta.

In [ ]:
contatore.dropna()

In [ ]:
contatore["kwh"].fillna(0)

In [ ]:
mediana = contatore["kwh"].median()
contatore["kwh"].fillna(mediana)

Il risultato non si salva da solo. `dropna`, `fillna` e quasi tutti i metodi di pandas
restituiscono una tabella nuova e lasciano l'originale com'è: `contatore` ha ancora i suoi
due `NaN`.

In [ ]:
contatore.dropna()
contatore

Per tenere il risultato lo riassegniamo, alla stessa variabile o a una nuova.

In [ ]:
contatore["kwh"] = contatore["kwh"].fillna(mediana)
contatore

Doppioni: `duplicated()` segna con `True` la seconda copia di una riga identica,
`drop_duplicates()` la toglie. Con `subset=["giorno"]` basta lo stesso giorno per dire
"identica", anche se il resto della riga cambia.

In [ ]:
contatore.duplicated()

In [ ]:
contatore = contatore.drop_duplicates()
contatore

Su un file grande il metodo è lo stesso. I prezzi dell'elettricità negli Stati Uniti, 85 mila
righe mensili per stato e settore, hanno la colonna `customers` vuota nei primi anni.

In [ ]:
prezzi = pd.read_csv("../Dati/U.S. Electricity Prices.csv")
prezzi.isna().sum()

`dropna(subset=[...])` butta una riga solo se manca quella colonna. Prima e dopo, `len`.

In [ ]:
prezzi_completi = prezzi.dropna(subset=["customers"])
print(f"prima: {len(prezzi)} righe, dopo: {len(prezzi_completi)} righe")

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Nel file dei prezzi la colonna `customers` (clienti serviti) manca nei primi anni. Crea
`customers_pieni`: la colonna con i mancanti sostituiti da 0. Poi conta in `n_mancanti`
quanti `NaN` restano.

</div>

In [ ]:
customers_pieni = prezzi["customers"].fillna(0)
n_mancanti = customers_pieni.isna().sum()
n_mancanti

In [ ]:
# Verifica: esegui senza modificare
assert n_mancanti == 0, "❌ Dopo fillna(0) non devono restare NaN"
assert len(customers_pieni) == len(prezzi), "❌ fillna non toglie righe: la lunghezza resta quella"
assert (customers_pieni == 0).sum() >= 26040, "❌ I 26040 mancanti devono essere diventati 0"
print("✅ Tutto corretto")

<a id="sez-2-1"></a>
### 2.1 Correggere un valore

Torniamo alle letture. Il customer care conferma: la lettura F1 di luglio del Panificio è
stata digitata con uno zero in più. Va divisa per 10. Prima la maschera, poi la correzione.

In [ ]:
mask_errata = letture["kwh"] > 5000
letture[mask_errata]

Questa cella non dà errore ma avvisa, e non cambia nulla: `letture[mask_errata]` è una copia,
e stiamo scrivendo sulla copia. Leggiamo l'avviso.

In [ ]:
letture[mask_errata]["kwh"] = letture[mask_errata]["kwh"] / 10
letture["kwh"].max()

`ChainedAssignmentError`: due operazioni in catena (prima filtra, poi assegna) non arrivano
mai all'originale. La forma giusta è una sola operazione, `loc[maschera, colonna] = valore`.

In [ ]:
letture.loc[mask_errata, "kwh"] = letture.loc[mask_errata, "kwh"] / 10
letture["kwh"].max()

Ora il massimo è 1434,9 kWh, lo Studio Dentistico a febbraio, e il Panificio a luglio è
tornato a 578,5. Da qui in avanti lavoriamo sulle letture corrette.

<div style="background:rgba(224,168,0,0.14); border-left:4px solid #e0a800; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**📌 Ricorda**

- `isna().sum()` prima di tutto.
- Il risultato non si salva da solo: `df = df.dropna()`.
- Si scrive con `df.loc[mask, "col"] = valore`, mai con `df[mask]["col"]`.

</div>

<a id="sez-3"></a>
## 3. Trasformare

Una colonna nuova si crea assegnando a un nome che non esiste ancora. Con l'aritmetica tra
colonne pandas lavora riga per riga da solo, senza cicli.

In [ ]:
letture["mwh"] = letture["kwh"] / 1000
letture.head(3)

Per il costo serve il prezzo della fascia. `.map` con un dizionario traduce ogni valore della
colonna nel suo corrispondente: fascia → €/kWh. Poi il prodotto, con `round` a due decimali.

In [ ]:
listino = {"F1": 0.21, "F2": 0.19, "F3": 0.17}

letture["eur_kwh"] = letture["fascia"].map(listino)
letture["costo"] = (letture["kwh"] * letture["eur_kwh"]).round(2)
letture.head(3)

Quando la regola è più di un conto, la scriviamo in una funzione e la applichiamo a ogni
valore con `.apply`. Qui una classe di consumo con due soglie.

In [ ]:
def classe_consumo(kwh):
    """Classifica una lettura mensile in basso, medio o alto."""
    if kwh < 300:
        return "basso"
    elif kwh < 1000:
        return "medio"
    return "alto"


letture["classe"] = letture["kwh"].apply(classe_consumo)
letture[["pod", "kwh", "classe"]].head()

Per una regola di una riga si può passare ad `apply` una `lambda`, senza dare un nome alla
funzione. Qui il numero della fascia, letto dal secondo carattere di `F1`, `F2`, `F3`. Se la
stessa cosa si fa con l'aritmetica tra colonne, meglio l'aritmetica: più veloce e più chiara.

In [ ]:
letture["fascia_num"] = letture["fascia"].apply(lambda f: int(f[1]))
letture[["fascia", "fascia_num"]].head(3)

Togliere una colonna: `drop(columns=[...])`. I MWh erano un esempio, non ci servono:
via, riassegnando.

In [ ]:
letture = letture.drop(columns=["mwh"])
letture.columns

Il file dei prezzi americani ha i nomi delle colonne in inglese e in camelCase. `rename` con
un dizionario vecchio → nuovo li sistema; le colonne non nominate restano come sono.

In [ ]:
prezzi = prezzi.rename(columns={"stateDescription": "stato", "sectorName": "settore", "price": "prezzo"})
prezzi.head(3)

`date` è testo, `2001-01-01` (le date vere arrivano nel prossimo notebook). I primi quattro
caratteri sono l'anno: `.str` applica a tutta la colonna le operazioni delle stringhe, e
`astype(int)` cambia il tipo da testo a numero intero.

In [ ]:
prezzi["anno"] = prezzi["date"].str[:4].astype(int)
prezzi[["date", "anno"]].dtypes

`sort_values` ordina per una colonna, `ascending=False` dal più grande. Con una lista di
colonne ordina per la prima e, a pari merito, per la seconda.

In [ ]:
letture.sort_values("kwh", ascending=False).head()

`set_index` e `reset_index` sono l'andata e il ritorno: una colonna diventa indice, l'indice
torna colonna. Lo rivedremo subito con `groupby`, che mette la chiave nell'indice.

In [ ]:
anagrafica_pod.reset_index().head(3)

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Aggiungi ad `anagrafica` la colonna `provincia` con `.map` e il dizionario `sigle`
(comune → sigla).

</div>

In [ ]:
sigle = {"Monza": "MB", "Lodi": "LO", "Cremona": "CR", "Cantù": "CO", "Lecco": "LC", "Varese": "VA"}

anagrafica["provincia"] = anagrafica["comune"].map(sigle)
anagrafica

In [ ]:
# Verifica: esegui senza modificare
assert list(anagrafica["provincia"]) == ["MB", "LO", "CR", "CO", "LC", "VA"], "❌ map va chiamato sulla colonna comune, con il dizionario sigle"
print("✅ Tutto corretto")

<div style="background:rgba(224,168,0,0.14); border-left:4px solid #e0a800; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**📌 Ricorda**

- Colonna nuova: `df["nuova"] = espressione sulle colonne`.
- Dizionario → `.map`; funzione → `.apply`.
- `rename`, `drop`, `sort_values` restituiscono una tabella nuova: riassegna.

</div>

<a id="sez-4"></a>
## 4. Aggregare

La domanda tipica è un totale per qualcosa: per POD, per fascia, per mese. In Excel è una
tabella pivot; in pandas è `groupby`: dividi per chiave, calcola, rimetti insieme.

In [ ]:
totale_pod = letture.groupby("pod")["kwh"].sum()
totale_pod

Il risultato è una Series: i POD sono finiti nell'indice e la colonna `pod` è sparita. Per
riavere una tabella normale, con `pod` come colonna, `reset_index()`. Lo faremo quasi sempre.

In [ ]:
tabella_pod = totale_pod.reset_index()
tabella_pod

Più statistiche in un colpo: `.agg` con la lista dei nomi.

In [ ]:
statistiche_pod = letture.groupby("pod")["kwh"].agg(["mean", "max"])
statistiche_pod.round(1)

Due chiavi: la lista dentro `groupby`. Il totale per POD e fascia.

In [ ]:
per_fascia = letture.groupby(["pod", "fascia"])["kwh"].sum()
per_fascia.reset_index().head(6)

Chi consuma di più? `idxmax` restituisce l'etichetta dell'indice dove sta il massimo: per
questo lo chiamiamo sulla Series, prima di `reset_index`.

In [ ]:
totale_pod.idxmax()

Sulla colonna di un DataFrame, `idxmax` dà l'etichetta della riga, che poi leggiamo con `loc`.

In [ ]:
posizione = letture["kwh"].idxmax()
letture.loc[posizione, ["pod", "data", "fascia", "kwh"]]

La pivot di Excel esiste anche qui: `pivot_table` con righe, colonne, valore e funzione.
Stessi numeri del `groupby` a due chiavi, disposti in griglia.

In [ ]:
letture.pivot_table(index="pod", columns="fascia", values="kwh", aggfunc="sum").round(1)

Per contare quante volte compare ogni valore c'è `value_counts`: la risposta a "quanti ne ho
per tipo". Sulle classi di consumo e, nel file grande, sui settori.

In [ ]:
letture["classe"].value_counts()

In [ ]:
prezzi["settore"].value_counts()

Con 85 mila righe `groupby` fa lo stesso lavoro senza battere ciglio: il prezzo medio
(centesimi di dollaro al kWh) per settore e anno.

In [ ]:
prezzo_settore_anno = prezzi.groupby(["settore", "anno"])["prezzo"].mean()
prezzo_settore_anno.reset_index().head()

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Calcola la Series `costo_cliente`: il totale della colonna `costo` di `letture` per
`cliente` (senza `reset_index`, ci serve l'indice). Poi metti in `cliente_top` il cliente
che spende di più.

</div>

In [ ]:
costo_cliente = letture.groupby("cliente")["costo"].sum()
cliente_top = costo_cliente.idxmax()
print(cliente_top)
costo_cliente.round(2)

In [ ]:
# Verifica: esegui senza modificare
assert len(costo_cliente) == 6, "❌ Un totale per ciascuno dei sei clienti"
assert round(costo_cliente.sum(), 2) == 25083.22, "❌ Somma la colonna costo, non kwh"
assert cliente_top == "Studio Dentistico Colombo", "❌ cliente_top: idxmax sulla Series"
print("✅ Tutto corretto")

<div style="background:rgba(126,87,194,0.12); border-left:4px solid #7e57c2; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**📘 Approfondimento · Il valore del gruppo accanto a ogni riga (puoi saltarlo)**

`groupby(...).transform("median")` calcola la statistica per gruppo ma la restituisce lunga
quanto la tabella di partenza, un valore per riga: così ogni lettura si confronta con la
mediana del suo POD, senza merge.

```python
mediana_pod = letture.groupby("pod")["kwh"].transform("median")
letture["rapporto"] = (letture["kwh"] / mediana_pod).round(2)
```

Una lettura con `rapporto` 6 è da controllare; 1,3 è un inverno normale.

</div>

<div style="background:rgba(224,168,0,0.14); border-left:4px solid #e0a800; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**📌 Ricorda**

- `groupby(chiave)[colonna].funzione()`, poi `reset_index()`.
- `idxmax` prima di `reset_index`: vuole l'etichetta.
- `value_counts` per contare, `pivot_table` per la griglia.

</div>

<a id="sez-5"></a>
## 5. Unire

Le informazioni stanno quasi sempre in due tabelle: i consumi da una parte, l'anagrafica
dall'altra. `merge` le incrocia su una colonna in comune: un CERCA.VERT che porta tutte le
colonne in una volta.

Il foglio Consumi ha i kWh per POD e mese ma non dice chi è il cliente; l'Anagrafica sì.
Prima del merge contiamo le righe di entrambi.

In [ ]:
consumi = pd.read_excel("../Dati/bolletta_esempio.xlsx", sheet_name="Consumi")
print(f"consumi: {len(consumi)} righe, anagrafica: {len(anagrafica)} righe")
consumi.head(3)

In [ ]:
consumi_clienti = pd.merge(consumi, anagrafica, on="pod", how="left")
print(f"dopo il merge: {len(consumi_clienti)} righe")
consumi_clienti.head(3)

`on` è la colonna in comune. `how="left"` dice: tieni tutte le righe della tabella di
sinistra e cerca la corrispondenza a destra; chi non la trova resta, con `NaN` nelle colonne
nuove. È il comportamento di CERCA.VERT, e il più usato.

Vediamolo su una tabella piccola: tre offerte commerciali, una delle quali per un POD che
non è nostro. `how="inner"` tiene solo le corrispondenze, `how="outer"` tiene tutto da
entrambe le parti. Con `indicator=True` la colonna `_merge` dice da dove viene ogni riga: è
il modo più rapido per trovare chi è rimasto senza corrispondenza.

In [ ]:
offerte = pd.DataFrame({
    "pod": ["IT001E12345678", "IT001E45678901", "IT001E99999999"],
    "sconto_pct": [5, 10, 7],
})
tutto = pd.merge(anagrafica, offerte, on="pod", how="outer", indicator=True)
tutto[["pod", "cliente", "sconto_pct", "_merge"]]

In [ ]:
mask_senza_offerta = tutto["_merge"] == "left_only"
tutto[mask_senza_offerta]

Dal database: 20 clienti e 30 POD, legati da `id_cliente`. Un cliente può avere più POD, e
dopo il merge le righe saranno più di 20. Non è un errore, è la risposta giusta; ma va
previsto, perciò si conta prima e dopo.

In [ ]:
con = sqlite3.connect("../Dati/utility.db")
clienti_db = pd.read_sql("SELECT * FROM clienti", con)
pod_db = pd.read_sql("SELECT * FROM pod", con)
con.close()
print(f"clienti: {len(clienti_db)} righe, pod: {len(pod_db)} righe")

In [ ]:
clienti_pod = pd.merge(clienti_db, pod_db, on="id_cliente", how="left")
print(f"dopo il merge: {len(clienti_pod)} righe")
clienti_pod.head()

<div style="background:rgba(217,83,79,0.12); border-left:4px solid #d9534f; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**⚠️ Attenzione**

Se la chiave non è unica nella tabella di destra, le righe si moltiplicano: 20 clienti
diventano 34 righe perché alcuni hanno più POD. Se dopo un merge le righe aumentano senza un
motivo, la chiave va controllata con `duplicated()` prima di andare avanti.

</div>

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Fai il merge al contrario: `pod_db` a sinistra, `clienti_db` a destra, su `id_cliente`,
`how="left"`, in `pod_clienti`. Quante righe ti aspetti?

</div>

In [ ]:
pod_clienti = pd.merge(pod_db, clienti_db, on="id_cliente", how="left")
print(len(pod_clienti))
pod_clienti.head()

In [ ]:
# Verifica: esegui senza modificare
assert len(pod_clienti) == 30, "❌ Ogni POD ha un solo cliente: le righe restano 30"
assert "ragione_sociale" in pod_clienti.columns, "❌ Dopo il merge devono esserci le colonne di clienti_db"
assert pod_clienti["ragione_sociale"].isna().sum() == 0, "❌ Tutti i POD devono aver trovato il loro cliente"
print("✅ Tutto corretto")

<a id="sez-5-1"></a>
### 5.1 Impilare tabelle: concat

`concat` mette una tabella sotto l'altra: stesse colonne, più righe. È il caso dei file
mensili dello stesso fornitore, o dei clienti nuovi che arrivano dal commerciale.
`ignore_index=True` rinumera le righe da 0.

In [ ]:
nuovi_clienti = pd.DataFrame({
    "pod": ["IT001E78901234", "IT001E89012345"],
    "cliente": ["Gelateria Polo Nord", "Palestra Olimpo"],
    "comune": ["Como", "Bergamo"],
})
anagrafica_completa = pd.concat([anagrafica, nuovi_clienti], ignore_index=True)
anagrafica_completa

Se una delle due tabelle ha una colonna in più, le righe dell'altra la ricevono vuota
(`NaN`): `concat` allinea per nome di colonna, non per posizione.

<div style="background:rgba(140,140,140,0.12); border-left:4px solid #8c8c8c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**💡 Nota**

`merge` e `concat` lavorano sulle colonne. Esiste anche `df.join(altro)`, che unisce
sull'indice: capita di incontrarlo nel codice degli altri, ma `merge` fa lo stesso lavoro e si
legge meglio.

</div>

<div style="background:rgba(224,168,0,0.14); border-left:4px solid #e0a800; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**📌 Ricorda**

- Prima e dopo il merge: `len()`.
- `how="left"` è il CERCA.VERT; `indicator=True` per vedere chi resta senza corrispondenza.
- `concat` per impilare, `merge` per incrociare.

</div>

<a id="sez-6"></a>
## 6. Data Wrangler

Data Wrangler è un'estensione di VS Code che mostra un DataFrame come un foglio di calcolo e
traduce i clic in codice pandas. Serve per esplorare in fretta e per imparare: ogni
operazione fatta con il mouse mostra la riga di pandas che la fa.

Si apre da un DataFrame già in memoria. Dopo aver eseguito una cella che lo mostra, sotto
l'output compare il pulsante **Open in Data Wrangler**; in alternativa, nel pannello
**Variables** della barra del notebook, ogni DataFrame ha l'icona di Data Wrangler accanto.

Dentro: in alto le colonne con un riassunto (distribuzione, mancanti), a destra la lista dei
passi fatti, in basso il codice che li produce. Le operazioni si scelgono dal pannello
**Operations** a sinistra.

Proviamone due: **Filter** sulla colonna `fascia` uguale a `F1`, poi **Drop columns** su
`cliente`. Ogni passo si vede in anteprima e si può annullare.

Il pulsante **Export to notebook** incolla nel notebook una cella come questa: una funzione
con i passi, applicata a una copia del DataFrame.

In [ ]:
# codice incollato da Data Wrangler: i commenti in inglese sono suoi
def clean_data(letture):
    # Filter rows based on column: 'fascia'
    letture = letture[letture["fascia"] == "F1"]
    # Drop column: 'cliente'
    letture = letture.drop(columns=["cliente"])
    return letture


letture_clean = clean_data(letture.copy())
letture_clean.head()

Leggiamolo: è lo stesso pandas di questo notebook, con maschera e filtro sulla stessa riga e
`.copy()` per non toccare l'originale. Data Wrangler è un buon modo per scoprire il metodo
che non ricordiamo; il codice che produce va letto, e se serve accorciato.

<a id="sez-7"></a>
## 7. Esercizi

<a id="es-8-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 8.1 · Il POD con la lettura sballata**

Il fornitore ha rimandato il file delle letture, quello grezzo. Il controllo qualità è stanco
di cercare a occhio la lettura digitata male e vuole una procedura che la trovi da sola:
quella che, in fascia F1, è fuori scala rispetto alle altre letture dello stesso POD.

**Cosa fare.**

1. Rileggi `../Dati/letture_pod_2025.csv` in `letture` (separatore, decimale e codifica
   giusti) e tieni in `f1` solo le righe della fascia F1.
2. Calcola `statistiche`: per ogni POD il massimo e la mediana di `kwh`, con `groupby` e
   `.agg(["max", "median"])`.
3. Aggiungi a `statistiche` la colonna `rapporto`: massimo diviso mediana.
4. Metti in `pod_anomalo` il POD con il rapporto più alto e in `mese_anomalo` la `data`
   della riga di `f1` con il `kwh` più alto.

*Suggerimento:* `idxmax` su una colonna di `f1` dà l'etichetta della riga; con `loc` leggi la sua `data`.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 8.1**

La mediana non si lascia trascinare dal valore sbagliato, la media sì: il rapporto con la mediana isola l'anomalia (6,5 contro 1,2-1,4 degli altri POD).

</div>

In [ ]:
letture = pd.read_csv("../Dati/letture_pod_2025.csv", sep=";", decimal=",", encoding="latin-1")
mask_f1 = letture["fascia"] == "F1"
f1 = letture[mask_f1]

statistiche = f1.groupby("pod")["kwh"].agg(["max", "median"])
statistiche["rapporto"] = statistiche["max"] / statistiche["median"]

pod_anomalo = statistiche["rapporto"].idxmax()
mese_anomalo = f1.loc[f1["kwh"].idxmax(), "data"]
print(f"POD {pod_anomalo}, lettura del {mese_anomalo}")
statistiche.round(2)

In [ ]:
# Verifica: esegui senza modificare
assert len(f1) == 72, "❌ f1 deve contenere solo le 72 righe della fascia F1"
assert round(statistiche["rapporto"].max(), 1) == 6.5, "❌ rapporto: massimo diviso mediana, per POD"
assert pod_anomalo == "IT001E45678901", "❌ pod_anomalo: idxmax sulla colonna rapporto"
assert mese_anomalo == "01/07/2025", "❌ mese_anomalo: la data della riga di f1 con il kwh massimo"
print("✅ Esercizio 8.1 completato")

<a id="es-8-1-bis"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 8.1 bis · I prezzi del Texas**

*In alternativa all'8.1: stesso obiettivo, scenario diverso.*

Il commerciale prepara un confronto internazionale e vuole sapere come è cambiato il
prezzo residenziale in Texas, anno per anno. Il file è quello dei prezzi americani: 85 mila
righe, prezzi in centesimi di dollaro al kWh.

**Cosa fare.**

1. Leggi `../Dati/U.S. Electricity Prices.csv` in `prezzi` e crea la colonna `anno`
   (numero intero) dai primi quattro caratteri di `date`.
2. Tieni in `texas` le righe con `stateDescription` uguale a `Texas` e `sectorName`
   uguale a `residential`.
3. Calcola la Series `prezzo_anno`: media di `price` per `anno`, senza `reset_index`.
4. Metti in `anno_max` l'anno con il prezzo medio più alto e in `prezzo_max` quel prezzo,
   arrotondato a due decimali.

*Suggerimento:* Due condizioni, tra parentesi, unite da `&`. Il massimo di una Series è `.max()`, la sua etichetta `.idxmax()`.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 8.1 bis**

L'anno estratto con `.str[:4]` e convertito in intero basta per raggruppare; le date vere arrivano nel prossimo notebook.

</div>

In [ ]:
prezzi = pd.read_csv("../Dati/U.S. Electricity Prices.csv")
prezzi["anno"] = prezzi["date"].str[:4].astype(int)

mask_texas = (prezzi["stateDescription"] == "Texas") & (prezzi["sectorName"] == "residential")
texas = prezzi[mask_texas]

prezzo_anno = texas.groupby("anno")["price"].mean()
anno_max = prezzo_anno.idxmax()
prezzo_max = round(prezzo_anno.max(), 2)
print(f"Anno più caro: {anno_max} ({prezzo_max} cent/kWh)")
prezzo_anno.round(2)

In [ ]:
# Verifica: esegui senza modificare
assert len(texas) == 277, "❌ texas: stato Texas e settore residential, due condizioni con &"
assert len(prezzo_anno) == 24, "❌ prezzo_anno: una media per ciascuno dei 24 anni"
assert anno_max == 2023, "❌ anno_max: idxmax sulla Series prezzo_anno"
assert prezzo_max == 14.37, "❌ prezzo_max: il massimo di prezzo_anno, con due decimali"
print("✅ Esercizio 8.1 bis completato")

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Un passo in più (facoltativo)**

Il commerciale ora vuole tutti gli stati: una griglia stato × anno con il prezzo medio
residenziale. Riparti da `prezzi` con la colonna `anno`: tieni in `residenziale` il
settore `residential`, poi costruisci `tabella` con `pivot_table` (`index="stateDescription"`,
`columns="anno"`, `values="price"`, `aggfunc="mean"`), arrotondata a due decimali.
Infine `stato_piu_caro_2024`: lo stato con il prezzo più alto nella colonna `2024`.

</div>

In [ ]:
mask_res = prezzi["sectorName"] == "residential"
residenziale = prezzi[mask_res]

tabella = residenziale.pivot_table(index="stateDescription", columns="anno", values="price", aggfunc="mean").round(2)
stato_piu_caro_2024 = tabella[2024].idxmax()
print(stato_piu_caro_2024)
tabella.head()

In [ ]:
# Verifica: esegui senza modificare
assert tabella.shape == (62, 24), "❌ tabella: 62 stati in riga, 24 anni in colonna"
assert round(tabella.loc["Hawaii", 2024], 2) == 44.28, "❌ tabella: media di price, con anno intero in colonna"
assert stato_piu_caro_2024 == "Hawaii", "❌ stato_piu_caro_2024: idxmax sulla colonna 2024 della tabella"
print("✅ Esercizio 8.1 bis · un passo in più completato")

<a id="es-8-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 8.2 · Bolletta per cliente**

Fine anno: la fatturazione vuole, per ogni cliente, il costo dell'energia 2025 a partire
dai tre fogli di `bolletta_esempio.xlsx`. Consumi ha i kWh per POD e mese, una colonna per
fascia; Listino i prezzi per fascia; Anagrafica chi è il cliente. Oggi lo fa un collega
con tre CERCA.VERT e una pivot, e il risultato deve finire in un Excel.

**Cosa fare.**

1. Leggi i tre fogli in `consumi`, `listino`, `anagrafica`.
2. Metti `fascia` come indice del listino e leggi con `loc` i tre prezzi `p_f1`, `p_f2`,
   `p_f3`.
3. Aggiungi a `consumi` la colonna `costo`: `F1` per `p_f1`, più `F2` per `p_f2`, più `F3`
   per `p_f3`.
4. Calcola `costo_pod`: il totale di `costo` per POD, arrotondato a due decimali, con
   `reset_index`.
5. Unisci `costo_pod` con `anagrafica` in `bolletta` (merge su `pod`, `how="left"`),
   stampando `len` prima e dopo.
6. Salva `bolletta` in `bolletta_2025.xlsx` nella cartella del notebook, senza indice.

*Suggerimento:* `sheet_name=None` legge tutti i fogli in un dizionario; `to_excel(nome, index=False)` scrive il file.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 8.2**

Il listino ha tre righe: metterle nell'indice e leggerle con `loc` è più chiaro di un merge su una tabella larga. Il merge serve dove serve, per attaccare l'anagrafica.

</div>

In [ ]:
fogli = pd.read_excel("../Dati/bolletta_esempio.xlsx", sheet_name=None)
consumi = fogli["Consumi"]
listino = fogli["Listino"]
anagrafica = fogli["Anagrafica"]

listino_fascia = listino.set_index("fascia")
p_f1 = listino_fascia.loc["F1", "eur_kwh"]
p_f2 = listino_fascia.loc["F2", "eur_kwh"]
p_f3 = listino_fascia.loc["F3", "eur_kwh"]

consumi["costo"] = consumi["F1"] * p_f1 + consumi["F2"] * p_f2 + consumi["F3"] * p_f3
costo_pod = consumi.groupby("pod")["costo"].sum()
costo_pod = costo_pod.round(2).reset_index()

print(f"prima: {len(costo_pod)} righe")
bolletta = pd.merge(costo_pod, anagrafica, on="pod", how="left")
print(f"dopo: {len(bolletta)} righe")
bolletta.to_excel("bolletta_2025.xlsx", index=False)
bolletta

In [ ]:
# Verifica: esegui senza modificare
assert len(bolletta) == 6, "❌ bolletta: una riga per POD, sei POD"
assert round(bolletta["costo"].sum(), 2) == 17874.14, "❌ costo: F1*p_f1 + F2*p_f2 + F3*p_f3, poi somma per POD"
assert "cliente" in bolletta.columns, "❌ bolletta: dopo il merge deve avere la colonna cliente dell'anagrafica"
assert len(pd.read_excel("bolletta_2025.xlsx")) == 6, "❌ Il file bolletta_2025.xlsx deve avere sei righe"
print("✅ Esercizio 8.2 completato")

<a id="es-8-2-bis"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 8.2 bis · Clienti senza letture**

*In alternativa all'8.2: stesso obiettivo, scenario diverso.*

Il customer care ha venti clienti nel database e il sospetto che per alcuni non arrivi
nessuna lettura. Nel database le letture sono per POD, e i POD sono legati al cliente da
`id_cliente`: due salti. Serve l'elenco dei clienti senza nemmeno una lettura a marzo.

**Cosa fare.**

1. Leggi dal database `../Dati/utility.db` le tre tabelle in `clienti_db`, `pod_db`,
   `letture_db` e chiudi la connessione.
2. Unisci `letture_db` con `pod_db` su `pod` (`how="left"`) in `letture_pod`: ogni lettura
   acquista il suo `id_cliente`. Controlla `len` prima e dopo.
3. Calcola `kwh_cliente`: totale di `kwh` per `id_cliente`, con `reset_index`.
4. Unisci `clienti_db` con `kwh_cliente` su `id_cliente`, `how="left"` e `indicator=True`,
   in `clienti_kwh`; tieni in `senza_letture` le righe con `_merge` uguale a `left_only`.

*Suggerimento:* Il merge con `indicator=True` aggiunge la colonna `_merge`: filtrala con una maschera come qualunque altra colonna.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 8.2 bis**

Il totale per cliente si calcola prima del merge con l'anagrafica: così la tabella finale ha una riga per cliente e `left_only` significa davvero "nessuna lettura".

</div>

In [ ]:
con = sqlite3.connect("../Dati/utility.db")
clienti_db = pd.read_sql("SELECT * FROM clienti", con)
pod_db = pd.read_sql("SELECT * FROM pod", con)
letture_db = pd.read_sql("SELECT * FROM letture", con)
con.close()

letture_pod = pd.merge(letture_db, pod_db, on="pod", how="left")
print(f"letture: {len(letture_db)} righe, dopo il merge: {len(letture_pod)} righe")

kwh_cliente = letture_pod.groupby("id_cliente")["kwh"].sum()
kwh_cliente = kwh_cliente.reset_index()
clienti_kwh = pd.merge(clienti_db, kwh_cliente, on="id_cliente", how="left", indicator=True)
mask_senza = clienti_kwh["_merge"] == "left_only"
senza_letture = clienti_kwh[mask_senza]
senza_letture

In [ ]:
# Verifica: esegui senza modificare
assert len(letture_pod) == 930, "❌ letture_pod: ogni POD ha un solo cliente, le 930 letture restano 930"
assert len(clienti_kwh) == 20, "❌ clienti_kwh: how='left' da clienti_db tiene tutti i 20 clienti"
assert len(senza_letture) == 4, "❌ senza_letture: le righe con _merge uguale a left_only"
assert set(senza_letture["ragione_sociale"]) == {"Panificio Rossi", "Lavanderia Splendor", "Pasticceria Dolce Vita", "Serra Fiorita"}, "❌ Controlla la direzione del merge: clienti_db a sinistra"
print("✅ Esercizio 8.2 bis completato")

---

**Fine del notebook 08.** Prossimo: [09 · pandas: le date](09_Pandas_date.ipynb)